In [ ]:
import tensorflow as tf
tf.config.list_physical_devices('GPU')

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

In [ ]:
#Connect Google Drive
from google.colab import drive
drive.mount('/gdrive')

In [ ]:
!pip install tensorflow opencv-python matplotlib

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

unzipping the file using the system unzip

In [ ]:
!unzip /content/aquatic_animals_dataset.zip -d /content/dataset

Archive:  /content/aquatic_animals_dataset.zip
  inflating: /content/dataset/aquatic_animals/crab/image_0.jpeg  
  inflating: /content/dataset/aquatic_animals/crab/image_1.jpeg  
  inflating: /content/dataset/aquatic_animals/crab/image_10.jpeg  
  inflating: /content/dataset/aquatic_animals/crab/image_11.jpeg  
  inflating: /content/dataset/aquatic_animals/crab/image_12.jpeg  
  inflating: /content/dataset/aquatic_animals/crab/image_13.jpeg  
  inflating: /content/dataset/aquatic_animals/crab/image_14.jpeg  
  inflating: /content/dataset/aquatic_animals/crab/image_15.jpeg  
  inflating: /content/dataset/aquatic_animals/crab/image_16.jpeg  
  inflating: /content/dataset/aquatic_animals/crab/image_17.jpeg  
  inflating: /content/dataset/aquatic_animals/crab/image_18.jpeg  
  inflating: /content/dataset/aquatic_animals/crab/image_19.jpeg  
  inflating: /content/dataset/aquatic_animals/crab/image_2.jpeg  
  inflating: /content/dataset/aquatic_animals/crab/image_20.jpeg  
  inflating: /cont

In [ ]:
import zipfile

zip_path = "/content/drive/MyDrive/Data_science_project/dataset/aquatic_animals_dataset.zip"
extract_path = "/content/drive/MyDrive/Data_science_project/dataset"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Extraction complete")

Extraction complete


In [ ]:
import os

print(os.listdir("/content/drive/MyDrive/Data_science_project/dataset/aquatic_animals"))

['crab', 'dolphin', 'octopus', 'seahorse', 'seal', 'seaturtle', 'shark', 'starfish']


checking the dataset

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

dataset_path = "/content/drive/MyDrive/Data_science_project/dataset/aquatic_animals"

img_size = (224, 224)
batch_size = 32

datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_data = datagen.flow_from_directory(
    dataset_path,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='training'
)

val_data = datagen.flow_from_directory(
    dataset_path,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation'
)

Found 320 images belonging to 8 classes.
Found 80 images belonging to 8 classes.


loading pre ptrained model

In [ ]:
from tensorflow.keras.applications import MobileNetV2

base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)

# Freeze the base model
base_model.trainable = False

building my model

In [ ]:
from tensorflow.keras import layers, models

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dense(train_data.num_classes, activation='softmax')
])

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 8)              │         1,032 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,984 (9.24 MB)

 Trainable params: 165,000 (644.53 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

compiling the model

In [ ]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

training model

In [ ]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=5
)

Epoch 1/5
10/10 ━━━━━━━━━━━━━━━━━━━━ 104s 10s/step - accuracy: 0.5562 - loss: 1.3088 - val_accuracy: 0.7250 - val_loss: 0.7180
Epoch 2/5
10/10 ━━━━━━━━━━━━━━━━━━━━ 20s 2s/step - accuracy: 0.9312 - loss: 0.2583 - val_accuracy: 0.7875 - val_loss: 0.5358
Epoch 3/5
10/10 ━━━━━━━━━━━━━━━━━━━━ 20s 2s/step - accuracy: 0.9625 - loss: 0.1051 - val_accuracy: 0.8125 - val_loss: 0.5145
Epoch 4/5
10/10 ━━━━━━━━━━━━━━━━━━━━ 22s 2s/step - accuracy: 1.0000 - loss: 0.0388 - val_accuracy: 0.8250 - val_loss: 0.4793
Epoch 5/5
10/10 ━━━━━━━━━━━━━━━━━━━━ 19s 2s/step - accuracy: 1.0000 - loss: 0.0231 - val_accuracy: 0.8250 - val_loss: 0.4947


testing model with different image

In [ ]:
from tensorflow.keras.preprocessing import image
import numpy as np

img_path = "/content/drive/MyDrive/Data_science_project/test_images/killer_whale.jpg"

img = image.load_img(img_path, target_size=(224,224))
img_array = image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0) / 255.0

prediction = model.predict(img_array)
class_names = list(train_data.class_indices.keys())

print("Prediction:", class_names[np.argmax(prediction)])

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
Prediction: dolphin


In [ ]:
import os

os.makedirs("/content/test_images", exist_ok=True)

In [ ]:
from tensorflow.keras.preprocessing import image
import numpy as np
import os

test_folder = "/content/drive/MyDrive/Data_science_project/test_images"

class_names = list(train_data.class_indices.keys())

for img_name in os.listdir(test_folder):
    img_path = os.path.join(test_folder, img_name)

    img = image.load_img(img_path, target_size=(224,224))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0) / 255.0

    prediction = model.predict(img_array)
    predicted_class = class_names[np.argmax(prediction)]

    print(f"{img_name} → {predicted_class}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 134ms/step
killer_whale.jpg → dolphin
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 150ms/step
starfish.jpg → starfish
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 135ms/step
octopus.jpg → octopus


In [ ]:
from tensorflow.keras.preprocessing import image
import numpy as np
import os

test_folder = "/content/drive/MyDrive/Data_science_project/test_images"
class_names = list(train_data.class_indices.keys())

for img_name in os.listdir(test_folder):
    img_path = os.path.join(test_folder, img_name)

    img = image.load_img(img_path, target_size=(224,224))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0) / 255.0

    prediction = model.predict(img_array)

    predicted_class = class_names[np.argmax(prediction)]
    confidence = np.max(prediction)

    print(f"{img_name} → {predicted_class} ({confidence:.2f})")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 98ms/step
killer_whale.jpg → dolphin (0.53)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step
starfish.jpg → starfish (1.00)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step
octopus.jpg → octopus (0.83)


In [ ]:
import os
os.path.exists('/content/drive')

True

saving the model

In [ ]:
model.save("/content/drive/MyDrive/Data_science_project/models/aquatic_model_v1.keras")

In [ ]:
model.save("/content/drive/MyDrive/Data_science_project/models/aquatic_model_v1.h5")

In [ ]:
import tensorflow as tf

# Load your existing model
model = tf.keras.models.load_model("/content/drive/MyDrive/Data_science_project/models/aquatic_model_v1.h5")

# Save as SavedModel format (Keras 3 way)
model.export("aquatic_savedmodel")

# Zip and download
import shutil
from google.colab import files

shutil.make_archive("aquatic_savedmodel", "zip", "aquatic_savedmodel")
files.download("aquatic_savedmodel.zip")

Saved artifact at 'aquatic_savedmodel'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='input_layer_2')
Output Type:
  TensorSpec(shape=(None, 8), dtype=tf.float32, name=None)
Captures:
  134921196050128: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134921164209040: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134921164206544: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134921196049744: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134921164209424: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134921164207888: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134921164208848: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134921164209616: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134921164210000: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134921164207504: TensorSpec(shape=(), dtype=tf.resource, name=None)
  13492116420

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>